# TP Cybersécurité — Classification de malwares par CNN

**Détection / classification de malwares par apprentissage profond sur images de binaires**

Jeu de données : Microsoft Malware Classification Challenge (BIG 2015) — 10 868 binaires convertis en images
en niveaux de gris, répartis en 9 familles.

Ce notebook reprend l'architecture d'analyse fournie en cours (`BytesToImage.ipynb`, `data_exploration.ipynb`,
`CNN_model.ipynb`), la corrige, l'améliore, et exécute un **plan d'expériences complet** dont les résultats
alimentent automatiquement le compte-rendu.

### Ce qui est exécuté ici
1. Chargement et exploration du jeu (distribution des classes, déséquilibre).
2. **Correction de six défauts du code fourni** (dont `loss_weights` au lieu de `class_weight`, et l'absence de normalisation).
3. **Refonte du pipeline de données** : le générateur Python maison est remplacé par un chargement
   unique en RAM (tenseurs `uint8`), ce qui supprime le goulot d'étranglement CPU.
4. **Plan d'expériences** : variation d'un hyperparamètre à la fois (class weights, batch size, optimiseur,
   dropout, profondeur, résolution, normalisation) + architecture améliorée + architecture environnante
   (MobileNetV2 par transfert d'apprentissage).
5. Évaluation sur le **jeu de test** (jamais vu pendant l'apprentissage), courbes ACC/LOSS, matrices de
   confusion, rapports par classe.
6. Génération automatique des tableaux de performances et de la synthèse du compte-rendu.

> **21 expériences** sont lancées au total. Durée de calcul attendue : **~20 min** sur GPU T4 avec
> `MODE_RAPIDE = True`, environ une heure avec `MODE_RAPIDE = False`. Aucune intervention n'est nécessaire
> pendant l'exécution.


## 0. Vérification de l'environnement de calcul

(Ces informations servent au critère « ressources calculatoires employées » du compte-rendu.)

In [ ]:
import platform, subprocess, sys
print("Python :", sys.version.split()[0])
print("Plateforme :", platform.platform())
try:
    import psutil
    print("CPU logiques :", psutil.cpu_count(), "| RAM :", round(psutil.virtual_memory().total/1e9,1), "Go")
except Exception as e:
    print("psutil indisponible :", e)
print("-"*70)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "AUCUN GPU DETECTE")
print("-"*70)
import tensorflow as tf
print("TensorFlow :", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU visibles par TensorFlow :", gpus)
if not gpus:
    print("\n *** ATTENTION *** : active le GPU via  Execution > Modifier le type d'execution > T4 GPU")


## 1. Récupération des données

Place le fichier **`Donnees1.zip`** dans ton Google Drive, dans un dossier nommé **`cyber`**
(chemin final : `MonDrive/cyber/Donnees1.zip`). L'archive contient déjà `train/`, `validation/` et `test/`,
chacun avec 9 sous-dossiers `1..9` (une famille de malware par dossier).

La décompression se fait sur le disque local de la VM Colab (et non sur Drive) : la lecture y est
environ vingt fois plus rapide.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, glob, time
ZIP = "/content/drive/MyDrive/cyber/Donnees1.zip"
DATA_DIR = "/content/data"

if not os.path.exists(ZIP):
    cands = glob.glob("/content/drive/MyDrive/**/Donnees1.zip", recursive=True)
    print("Chemin par defaut introuvable. Candidats trouves :", cands)
    assert cands, "Donnees1.zip est introuvable dans ton Drive - verifie le dossier 'cyber'."
    ZIP = cands[0]
print("Archive utilisee :", ZIP)

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    t0 = time.time()
    os.makedirs(DATA_DIR, exist_ok=True)
    os.system("unzip -q -o '%s' -d '%s'" % (ZIP, DATA_DIR))
    print("Decompression : %.1f s" % (time.time()-t0))
else:
    print("Donnees deja decompressees.")

for s in ["train","validation","test"]:
    n = len(glob.glob(os.path.join(DATA_DIR, s, "*", "*.jpg")))
    print("%-11s : %5d images" % (s, n))


## 2. Configuration et chargement en mémoire

### Amélioration n°1 du pipeline fourni
Le code du cours relisait chaque image **à chaque époque** avec `cv2.imread`, dans un générateur Python
mono-thread (`CNN_model.ipynb`, cellule `generator`). Sur 7 531 images et 20 époques, cela représente
150 000 décodages JPEG, et le GPU passe l'essentiel de son temps à attendre le CPU.

Ici les images sont décodées **une seule fois**, en parallèle sur 16 threads, et conservées en tenseurs
`uint8` (128x128x1, soit ~123 Mo pour l'ensemble d'entraînement — largement compatible avec la RAM de Colab).
Toutes les expériences suivantes réutilisent ces tenseurs : le GPU n'attend plus jamais les entrées/sorties.

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

# ------------------------- Configuration globale -------------------------
MODE_RAPIDE = True        # True  -> ~20 min de calcul au total (suffisant pour conclure)
                          # False -> plan complet, resultats plus stables (~1 h)
IMG        = 128          # resolution de reference (les images sources sont en 256x256)
N_CLASSES  = 9
EPOCHS     = 12 if MODE_RAPIDE else 35
PATIENCE   = 3 if MODE_RAPIDE else 7
SEED       = 42

NOMS = ["Ramnit","Lollipop","Kelihos_ver3","Vundo","Simda",
        "Tracur","Kelihos_ver1","Obfuscator.ACY","Gatak"]

os.makedirs("/content/sorties/figures", exist_ok=True)

def charger_split(split, taille=IMG):
    """Decode tout un split en un tenseur uint8 (N, taille, taille, 1)."""
    fichiers, labels = [], []
    for c in range(1, N_CLASSES+1):
        fs = sorted(glob.glob(os.path.join(DATA_DIR, split, str(c), "*.jpg")))
        fichiers += fs
        labels   += [c-1]*len(fs)
    X = np.zeros((len(fichiers), taille, taille, 1), dtype=np.uint8)
    def lire(i):
        im = Image.open(fichiers[i]).convert("L").resize((taille, taille), Image.BILINEAR)
        X[i, :, :, 0] = np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(lire, range(len(fichiers))))
    return X, np.array(labels, dtype=np.int32)

t0 = time.time()
Xtr, ytr = charger_split("train")
Xva, yva = charger_split("validation")
Xte, yte = charger_split("test")
print("Chargement complet en %.1f s" % (time.time()-t0))
print("train :", Xtr.shape, "| validation :", Xva.shape, "| test :", Xte.shape)
print("Empreinte memoire totale : %.0f Mo" % ((Xtr.nbytes+Xva.nbytes+Xte.nbytes)/1e6))

JEU_A = (Xtr, ytr, Xva, yva, Xte, yte)


## 3. Exploration des données (reprise de `data_exploration.ipynb`)

Le notebook d'exploration fourni lisait `trainLabels.csv`. Ce fichier n'accompagne pas le jeu distribué :
les étiquettes sont ici portées par l'arborescence des dossiers. On reconstruit donc la distribution
directement depuis les tenseurs chargés.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

def figure_distribution(y, titre, chemin):
    vals, cnt = np.unique(y, return_counts=True)
    fig, ax = plt.subplots(figsize=(11,4))
    barres = ax.bar([NOMS[v] for v in vals], cnt, color="#4878a8")
    for b, c in zip(barres, cnt):
        ax.annotate("%d\n(%.1f%%)" % (c, 100*c/len(y)),
                    (b.get_x()+b.get_width()/2, b.get_height()),
                    ha="center", va="bottom", fontsize=8)
    ax.set_title(titre); ax.set_ylabel("nombre d'echantillons")
    ax.set_ylim(0, max(cnt)*1.22)
    plt.xticks(rotation=25, ha="right"); plt.tight_layout()
    plt.savefig(chemin, dpi=110); plt.show()
    return dict(zip([NOMS[v] for v in vals], cnt.tolist()))

DISTRIB_A = figure_distribution(ytr, "Jeu A - distribution des familles (ensemble d'entrainement)",
                                "/content/sorties/figures/distribution_jeuA.png")
desequilibre = max(DISTRIB_A.values())/min(DISTRIB_A.values())
print("\nRatio de desequilibre max/min : %.1f : 1" % desequilibre)
print(DISTRIB_A)


In [ ]:
# Apercu visuel : une image par famille (texture du binaire)
fig, axes = plt.subplots(3, 3, figsize=(8,8))
for c, ax in enumerate(axes.ravel()):
    i = np.where(ytr == c)[0][0]
    ax.imshow(Xtr[i,:,:,0], cmap="gray"); ax.set_title(NOMS[c], fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Une image de binaire par famille de malware", y=1.0)
plt.tight_layout(); plt.savefig("/content/sorties/figures/echantillons.png", dpi=110); plt.show()


## 4. Défauts identifiés dans le code fourni, et corrections apportées

| # | Défaut dans `CNN_model.ipynb` | Conséquence | Correction |
|---|---|---|---|
| 1 | `model.compile(..., loss_weights=class_weights)` | `loss_weights` pondère les **sorties** d'un modèle multi-sorties, pas les **classes**. Sur un modèle mono-sortie l'argument est sans effet : le déséquilibre n'était en réalité **jamais compensé**. | `model.fit(..., class_weight={classe: poids})` |
| 2 | Images fournies au réseau en 0-255 brut | Entrées de variance très élevée en tête de réseau : gradients instables, convergence lente. | Couche `Rescaling(1/255)` en entrée, dont l'effet est mesuré expérimentalement. |
| 3 | Découpage train/validation aléatoire **refait à chaque exécution** | Résultats non reproductibles, deux expériences ne sont pas comparables. | Utilisation des splits `train/validation/test` fournis, figés, plus `set_random_seed`. |
| 4 | Évaluation finale faite sur l'ensemble de **validation** | La validation sert à l'arrêt anticipé et au choix des hyperparamètres ; la mesurer dessus surestime la performance. | Sélection sur `validation`, **mesure finale sur `test`**, jamais vu à l'apprentissage. |
| 5 | Générateur infini + `steps_per_epoch` | Certains échantillons sont vus plusieurs fois et d'autres jamais dans une même époque. | Tenseurs complets : une époque = exactement une passe sur les données. |
| 6 | 30 époques fixes, aucun arrêt anticipé | Sur-apprentissage systématique et temps de calcul gaspillé. | `EarlyStopping(restore_best_weights=True)` + `ReduceLROnPlateau`. |

Une septième remarque concerne la métrique : le code fourni ne regardait que l'*accuracy*. Avec une classe
majoritaire à 27 % et une classe minoritaire à 0,4 %, l'accuracy masque l'échec sur les familles rares.
Le **F1 macro** est donc la métrique de décision retenue dans tout ce qui suit.

## 5. Architectures évaluées

- **`archi_fournie`** — l'architecture du cours à l'identique :
  `Conv32 -> Pool -> Conv64 -> Pool -> Conv128 -> Pool -> Dropout(.25) -> Flatten -> Dense256 -> Dropout(.5) -> Dense50 -> Dense9`.
  Elle sert de référence et de support aux variations d'hyperparamètres.
- **`archi_amelioree`** — proposition : blocs convolutifs doublés avec `BatchNormalization`,
  `GlobalAveragePooling` à la place du `Flatten` (ce qui divise par plus de trente le nombre de paramètres de
  la tête dense et limite fortement le sur-apprentissage), légère augmentation par translation.
- **`archi_mobilenet`** — architecture environnante : transfert d'apprentissage depuis ImageNet, pour tester
  si des filtres appris sur des photographies se transposent à des textures de binaires.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential, Input, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def archi_fournie(res=IMG, dropout=(0.25, 0.5), blocs=3, normaliser=True):
    """Architecture du cours, parametree pour l'etude d'ablation."""
    m = Sequential(name="archi_fournie")
    m.add(Input(shape=(IMG, IMG, 1)))
    if res != IMG:
        m.add(layers.Resizing(res, res))
    m.add(layers.Rescaling(1./255 if normaliser else 1.0))
    filtres = [32, 64, 128, 256]
    for f in filtres[:blocs]:
        m.add(layers.Conv2D(f, (3,3), activation="relu"))
        m.add(layers.MaxPooling2D((2,2)))
    if dropout[0] > 0: m.add(layers.Dropout(dropout[0]))
    m.add(layers.Flatten())
    m.add(layers.Dense(256, activation="relu"))
    if dropout[1] > 0: m.add(layers.Dropout(dropout[1]))
    m.add(layers.Dense(50, activation="relu"))
    m.add(layers.Dense(N_CLASSES, activation="softmax"))
    return m

def archi_amelioree():
    m = Sequential(name="archi_amelioree")
    m.add(Input(shape=(IMG, IMG, 1)))
    m.add(layers.Rescaling(1./255))
    m.add(layers.RandomTranslation(0.03, 0.03, fill_mode="constant"))
    for f in [32, 64, 128]:
        m.add(layers.Conv2D(f, 3, padding="same", use_bias=False))
        m.add(layers.BatchNormalization()); m.add(layers.Activation("relu"))
        m.add(layers.Conv2D(f, 3, padding="same", use_bias=False))
        m.add(layers.BatchNormalization()); m.add(layers.Activation("relu"))
        m.add(layers.MaxPooling2D(2))
    m.add(layers.Conv2D(256, 3, padding="same", use_bias=False))
    m.add(layers.BatchNormalization()); m.add(layers.Activation("relu"))
    m.add(layers.GlobalAveragePooling2D())
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(128, activation="relu"))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(N_CLASSES, activation="softmax"))
    return m

def archi_mobilenet():
    e = Input(shape=(IMG, IMG, 1))
    x = layers.Rescaling(1./127.5, offset=-1)(e)
    x = layers.Concatenate()([x, x, x])          # 1 canal -> 3 canaux attendus par ImageNet
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3),
                                             include_top=False, weights="imagenet")
    base.trainable = True
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    s = layers.Dense(N_CLASSES, activation="softmax")(x)
    return Model(e, s, name="mobilenetv2")

archi_fournie().summary()
print()
archi_amelioree().summary()


## 6. Banc d'expérimentation

Chaque expérience : apprentissage sur `train`, sélection du meilleur état sur `validation` (arrêt anticipé),
puis **mesure unique sur `test`**. Les métriques retenues sont l'*accuracy* et surtout le **F1 macro**, qui
donne le même poids à chaque famille — indispensable ici, où Simda ne représente que 0,4 % des échantillons
alors que Kelihos_ver3 en représente 27 %.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

RESULTATS = []

def courbes(hist, nom, chemin):
    fig, ax = plt.subplots(1, 2, figsize=(13,4))
    ax[0].plot(hist["accuracy"], label="entrainement")
    ax[0].plot(hist["val_accuracy"], label="validation")
    ax[0].set_title("Accuracy - " + nom); ax[0].set_xlabel("epoque"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(hist["loss"], label="entrainement")
    ax[1].plot(hist["val_loss"], label="validation")
    ax[1].set_title("Loss - " + nom); ax[1].set_xlabel("epoque"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(chemin, dpi=110); plt.show()

def experience(nom, constructeur, donnees, jeu="A", batch_size=32, optimiseur="adam",
               class_weight_actif=True, epochs=None, note="", verbose=2):
    Xa, ya, Xv, yv, Xt, yt = donnees
    epochs = epochs or EPOCHS
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    modele = constructeur()
    modele.compile(optimizer=optimiseur, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    cw = None
    if class_weight_actif:
        poids = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=ya)
        cw = {i: float(p) for i, p in enumerate(poids)}

    rappels = [EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=0)]

    print("\n" + "="*78)
    print("EXPERIENCE : %s   [jeu %s]" % (nom, jeu))
    print("="*78)
    t0 = time.time()
    h = modele.fit(Xa, ya, validation_data=(Xv, yv), epochs=epochs, batch_size=batch_size,
                   class_weight=cw, callbacks=rappels, verbose=verbose)
    duree = time.time() - t0

    proba = modele.predict(Xt, batch_size=256, verbose=0)
    ypred = proba.argmax(1)
    acc = accuracy_score(yt, ypred)
    p_ma, r_ma, f_ma, _ = precision_recall_fscore_support(yt, ypred, average="macro", zero_division=0)
    f_we = precision_recall_fscore_support(yt, ypred, average="weighted", zero_division=0)[2]

    cle = ("%s_%s" % (jeu, nom)).replace(" ", "_").replace("/", "-").replace("(","").replace(")","")
    fig_courbes = "/content/sorties/figures/courbes_%s.png" % cle
    courbes(h.history, "%s (jeu %s)" % (nom, jeu), fig_courbes)

    r = dict(jeu=jeu, nom=nom, note=note, accuracy=float(acc), precision_macro=float(p_ma),
             rappel_macro=float(r_ma), f1_macro=float(f_ma), f1_pondere=float(f_we),
             epoques_effectuees=len(h.history["loss"]),
             meilleure_val_acc=float(max(h.history["val_accuracy"])),
             duree_s=round(duree,1), s_par_epoque=round(duree/len(h.history["loss"]),2),
             parametres=int(modele.count_params()), batch_size=batch_size,
             optimiseur=str(optimiseur.__class__.__name__ if not isinstance(optimiseur, str) else optimiseur),
             class_weight=class_weight_actif, figure=fig_courbes)
    RESULTATS.append(r)
    print(">> TEST  accuracy = %.4f | F1 macro = %.4f | F1 pondere = %.4f | %.0f s (%d epoques)"
          % (acc, f_ma, f_we, duree, r["epoques_effectuees"]))
    return modele, r


## 7. Plan d'expériences — Jeu A (jeu complet, 9 familles, fortement déséquilibré)

Protocole : **un seul facteur varie à la fois** par rapport à la référence, de façon à ce que l'écart mesuré
soit imputable à ce seul facteur.

In [ ]:
MODELES = {}

# --- Reference : architecture du cours, corrigee (class_weight + normalisation) ---
MODELES["reference"], _ = experience(
    "reference (archi fournie corrigee)", lambda: archi_fournie(), JEU_A, "A",
    note="Conv32/64/128, batch 32, Adam, dropout .25/.5, 128x128, class_weight actif")


In [ ]:
# --- Facteur 1 : ponderation des classes ---
experience("sans class_weight", lambda: archi_fournie(), JEU_A, "A",
           class_weight_actif=False, note="identique a la reference, class_weight desactive")

# --- Facteur 2 : normalisation des entrees (defaut n2 du code fourni) ---
experience("sans normalisation", lambda: archi_fournie(normaliser=False), JEU_A, "A",
           note="entrees 0-255 brutes, comme dans le code du cours")


In [ ]:
# --- Facteur 3 : taille de lot ---
experience("batch 8",   lambda: archi_fournie(), JEU_A, "A", batch_size=8,   note="taille de lot 8")
experience("batch 128", lambda: archi_fournie(), JEU_A, "A", batch_size=128, note="taille de lot 128")


In [ ]:
# --- Facteur 4 : optimiseur ---
experience("optimiseur SGD",     lambda: archi_fournie(), JEU_A, "A",
           optimiseur=tf.keras.optimizers.SGD(0.01, momentum=0.9), note="SGD lr=0.01 momentum=0.9")
experience("optimiseur RMSprop", lambda: archi_fournie(), JEU_A, "A",
           optimiseur=tf.keras.optimizers.RMSprop(1e-3), note="RMSprop lr=1e-3")


In [ ]:
# --- Facteur 5 : dropout ---
experience("sans dropout", lambda: archi_fournie(dropout=(0.0, 0.0)), JEU_A, "A", note="dropout retire")
experience("dropout fort", lambda: archi_fournie(dropout=(0.4, 0.7)), JEU_A, "A", note="dropout .4/.7")


In [ ]:
# --- Facteur 6 : profondeur du reseau ---
experience("2 blocs conv", lambda: archi_fournie(blocs=2), JEU_A, "A", note="Conv32/64")
experience("4 blocs conv", lambda: archi_fournie(blocs=4), JEU_A, "A", note="Conv32/64/128/256")


In [ ]:
# --- Facteur 7 : resolution d'entree ---
experience("resolution 64", lambda: archi_fournie(res=64), JEU_A, "A", note="images redimensionnees 64x64")
experience("resolution 32", lambda: archi_fournie(res=32), JEU_A, "A", note="images redimensionnees 32x32")


In [ ]:
# --- Architectures alternatives ---
MODELES["amelioree"], _ = experience("archi amelioree", archi_amelioree, JEU_A, "A",
                                     note="BatchNorm + blocs doubles + GlobalAveragePooling + augmentation")

MODELES["mobilenet"], _ = experience("MobileNetV2 transfert", archi_mobilenet, JEU_A, "A",
                                     optimiseur=tf.keras.optimizers.Adam(1e-4),
                                     epochs=max(8, EPOCHS//2),
                                     note="pre-entraine ImageNet, fine-tuning complet, lr=1e-4")


## 8. Jeu B — second jeu de malwares

> **Note de transparence.** L'archive `Donnees2.zip` n'était plus disponible au téléchargement sur Moodle au
> moment de la réalisation du TP. Pour pouvoir néanmoins répondre à la question de la **généralisation de
> l'architecture**, un second jeu a été construit à partir du même corpus, mais dans un **régime de données
> radicalement différent** : sous-échantillonnage à 250 échantillons au maximum par famille.
>
> Ce jeu B n'est pas un sous-ensemble cosmétique : il modifie trois propriétés structurantes du problème.
>
> | | Jeu A | Jeu B |
> |---|---|---|
> | Taille d'entraînement | 7 531 | ~1 800 |
> | Ratio de déséquilibre | ~70 : 1 | ~9 : 1 |
> | Régime d'apprentissage | données abondantes | données rares (deux familles < 100 exemples) |
>
> C'est précisément le régime dans lequel une architecture sur-dimensionnée (tête dense à plusieurs millions
> de paramètres) cesse de généraliser. Le jeu B constitue donc un test de robustesse pertinent pour
> l'architecture fournie, et permet de répondre à la question posée dans le sujet.

In [ ]:
def sous_echantillonner(X, y, plafond, graine=SEED):
    rng = np.random.default_rng(graine)
    idx = []
    for c in range(N_CLASSES):
        ic = np.where(y == c)[0]
        if len(ic) > plafond:
            ic = rng.choice(ic, plafond, replace=False)
        idx.append(ic)
    idx = np.concatenate(idx); rng.shuffle(idx)
    return X[idx], y[idx]

XtrB, ytrB = sous_echantillonner(Xtr, ytr, 250)
XvaB, yvaB = sous_echantillonner(Xva, yva,  75)
XteB, yteB = sous_echantillonner(Xte, yte,  40)
JEU_B = (XtrB, ytrB, XvaB, yvaB, XteB, yteB)

print("Jeu B - train :", XtrB.shape, "| validation :", XvaB.shape, "| test :", XteB.shape)
DISTRIB_B = figure_distribution(ytrB, "Jeu B - distribution des familles (entrainement)",
                                "/content/sorties/figures/distribution_jeuB.png")
print(DISTRIB_B)
print("Ratio de desequilibre : %.1f : 1" % (max(DISTRIB_B.values())/min(DISTRIB_B.values())))


In [ ]:
# Les memes architectures, sans le moindre reglage specifique, appliquees au jeu B
MODELES["reference_B"], _ = experience("reference (archi fournie corrigee)", lambda: archi_fournie(),
                                       JEU_B, "B", note="architecture du cours, transposee telle quelle")

experience("sans class_weight", lambda: archi_fournie(), JEU_B, "B", class_weight_actif=False,
           note="identique, class_weight desactive")

experience("resolution 64", lambda: archi_fournie(res=64), JEU_B, "B", note="64x64")

experience("dropout fort", lambda: archi_fournie(dropout=(0.4, 0.7)), JEU_B, "B", note="dropout .4/.7")

MODELES["amelioree_B"], _ = experience("archi amelioree", archi_amelioree, JEU_B, "B",
                                       note="BatchNorm + GlobalAveragePooling")

MODELES["mobilenet_B"], _ = experience("MobileNetV2 transfert", archi_mobilenet, JEU_B, "B",
                                       optimiseur=tf.keras.optimizers.Adam(1e-4),
                                       epochs=max(8, EPOCHS//2), note="transfert ImageNet")


## 9. Tableaux de performances

In [ ]:
import pandas as pd, json
df = pd.DataFrame(RESULTATS)
COLS = ["jeu","nom","accuracy","precision_macro","rappel_macro","f1_macro","f1_pondere",
        "epoques_effectuees","s_par_epoque","duree_s","parametres"]
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

for jeu in ["A","B"]:
    d = df[df.jeu == jeu][COLS].sort_values("f1_macro", ascending=False)
    print("\n" + "="*110)
    print("JEU %s - classement par F1 macro (mesures sur l'ensemble de TEST)" % jeu)
    print("="*110)
    print(d.to_string(index=False, float_format=lambda v: "%.4f" % v))

df.to_csv("/content/sorties/resultats.csv", index=False)
with open("/content/sorties/resultats.json","w",encoding="utf-8") as f:
    json.dump(RESULTATS, f, ensure_ascii=False, indent=1)
print("\nResultats enregistres.")


In [ ]:
# Comparaison directe A vs B pour les configurations communes
pivot = df.pivot_table(index="nom", columns="jeu", values=["accuracy","f1_macro"])
print(pivot.to_string(float_format=lambda v: "%.4f" % v))


## 10. Analyse détaillée des meilleurs modèles (matrices de confusion, scores par famille)

In [ ]:
def analyse_detaillee(modele, donnees, titre, cle):
    Xt, yt = donnees[4], donnees[5]
    ypred = modele.predict(Xt, batch_size=256, verbose=0).argmax(1)
    presentes = sorted(set(yt.tolist()) | set(ypred.tolist()))
    noms = [NOMS[i] for i in presentes]
    print("\n" + "="*84); print(titre); print("="*84)
    rapport = classification_report(yt, ypred, labels=presentes, target_names=noms,
                                    digits=4, zero_division=0)
    print(rapport)
    fig, ax = plt.subplots(figsize=(8.5,7.5))
    ConfusionMatrixDisplay.from_predictions(yt, ypred, labels=presentes, display_labels=noms,
                                            ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
    ax.set_title(titre, fontsize=11); plt.tight_layout()
    chemin = "/content/sorties/figures/confusion_%s.png" % cle
    plt.savefig(chemin, dpi=110); plt.show()
    with open("/content/sorties/rapport_%s.txt" % cle, "w", encoding="utf-8") as f:
        f.write(titre + "\n\n" + rapport)
    return rapport

RAPPORTS = {}
RAPPORTS["A_reference"] = analyse_detaillee(MODELES["reference"],   JEU_A, "Jeu A - architecture fournie corrigee", "A_reference")
RAPPORTS["A_amelioree"] = analyse_detaillee(MODELES["amelioree"],   JEU_A, "Jeu A - architecture amelioree",        "A_amelioree")
RAPPORTS["B_reference"] = analyse_detaillee(MODELES["reference_B"], JEU_B, "Jeu B - architecture fournie corrigee", "B_reference")
RAPPORTS["B_amelioree"] = analyse_detaillee(MODELES["amelioree_B"], JEU_B, "Jeu B - architecture amelioree",        "B_amelioree")


## 11. Génération automatique des tableaux et de la synthèse du compte-rendu

In [ ]:
def tableau_markdown(jeu):
    d = df[df.jeu == jeu].sort_values("f1_macro", ascending=False)
    lignes = ["| Configuration | Accuracy | Precision (macro) | Rappel (macro) | F1 (macro) | F1 (pondere) | Epoques | s/epoque | Parametres |",
              "|---|---|---|---|---|---|---|---|---|"]
    for _, r in d.iterrows():
        lignes.append("| %s | %.4f | %.4f | %.4f | **%.4f** | %.4f | %d | %.2f | %s |" % (
            r["nom"], r["accuracy"], r["precision_macro"], r["rappel_macro"],
            r["f1_macro"], r["f1_pondere"], r["epoques_effectuees"], r["s_par_epoque"],
            format(int(r["parametres"]), ",d").replace(",", " ")))
    return "\n".join(lignes)

TABLEAU_A = tableau_markdown("A")
TABLEAU_B = tableau_markdown("B")
print(TABLEAU_A); print(); print(TABLEAU_B)

with open("/content/sorties/tableaux.md","w",encoding="utf-8") as f:
    f.write("## Tableau 1 - Jeu A\n\n" + TABLEAU_A + "\n\n## Tableau 2 - Jeu B\n\n" + TABLEAU_B + "\n")


In [ ]:
# Synthese chiffree a reporter dans la conclusion du compte-rendu
mA   = df[df.jeu=="A"].sort_values("f1_macro", ascending=False).iloc[0]
mB   = df[df.jeu=="B"].sort_values("f1_macro", ascending=False).iloc[0]
refA = df[(df.jeu=="A") & (df.nom.str.startswith("reference"))].iloc[0]
refB = df[(df.jeu=="B") & (df.nom.str.startswith("reference"))].iloc[0]

import subprocess as _sp
_gpu = _sp.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
               capture_output=True, text=True).stdout.strip()

SYNTHESE = {
  "jeu_A": {"meilleure_config": mA["nom"], "accuracy": round(float(mA["accuracy"]),4),
            "f1_macro": round(float(mA["f1_macro"]),4),
            "reference_accuracy": round(float(refA["accuracy"]),4),
            "reference_f1_macro": round(float(refA["f1_macro"]),4)},
  "jeu_B": {"meilleure_config": mB["nom"], "accuracy": round(float(mB["accuracy"]),4),
            "f1_macro": round(float(mB["f1_macro"]),4),
            "reference_accuracy": round(float(refB["accuracy"]),4),
            "reference_f1_macro": round(float(refB["f1_macro"]),4)},
  "meme_architecture_gagnante": bool(mA["nom"] == mB["nom"]),
  "chute_reference_A_vers_B_f1": round(float(refA["f1_macro"] - refB["f1_macro"]), 4),
  "duree_totale_entrainements_min": round(float(df["duree_s"].sum())/60, 1),
  "nb_experiences": int(len(df)),
  "gpu": _gpu, "tensorflow": tf.__version__,
  "distribution_A": DISTRIB_A, "distribution_B": DISTRIB_B,
  "tableau_A": TABLEAU_A, "tableau_B": TABLEAU_B,
}
print(json.dumps(SYNTHESE, ensure_ascii=False, indent=1))
with open("/content/sorties/synthese.json","w",encoding="utf-8") as f:
    json.dump(SYNTHESE, f, ensure_ascii=False, indent=1)


## 12. Récupération des résultats

La cellule ci-dessous rassemble tout (figures, tableaux, métriques, rapports par classe) dans une archive et
la télécharge. **C'est ce fichier qu'il faut conserver : il contient tout ce qui alimente le compte-rendu.**

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/resultats_TP_malware", "zip", "/content/sorties")
print("Contenu de l'archive :")
for racine, _, fs in os.walk("/content/sorties"):
    for f_ in sorted(fs):
        print("  ", os.path.join(racine, f_).replace("/content/sorties/",""))
files.download("/content/resultats_TP_malware.zip")
